# Vzorové řešení — `pd.cut` a `pd.crosstab`

Tady najdeš jedno z možných řešení každého cvičení. **Hranice intervalů, popisky a způsob agregace nejsou jediné správné** — pokud jsi zvolila trochu jiné, ale logika sedí, je to v pořádku. K některým výsledkům jsou pod tabulkou komentáře, aby bylo jasné, co z dat plyne.

Společný setup:

In [1]:
import pandas as pd
import seaborn as sns

---
## 1. 🍽️ Tips

In [2]:
tips = sns.load_dataset('tips')
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


### A) Binning ceny účtu

In [3]:
# float('inf') = nekonečno, ať zachytíme i účty nad 30 $
bins = [0, 15, 30, float('inf')]
labels = ['levný účet', 'střední účet', 'drahý účet']
tips['bill_category'] = pd.cut(tips['total_bill'], bins=bins, labels=labels)

tips['bill_category'].value_counts().sort_index()

bill_category
levný účet       80
střední účet    132
drahý účet       32
Name: count, dtype: int64

### B) Procento spropitného + binning

In [4]:
# Nejdřív spočítáme procento spropitného z účtu
tips['tip_pct'] = tips['tip'] / tips['total_bill'] * 100

# A teď bin
bins_pct = [0, 10, 20, float('inf')]
labels_pct = ['skoupý', 'standard', 'štědrý']
tips['tip_pct_category'] = pd.cut(tips['tip_pct'], bins=bins_pct, labels=labels_pct)

tips['tip_pct_category'].value_counts().sort_index()

tip_pct_category
skoupý       27
standard    178
štědrý       39
Name: count, dtype: int64

### C) Crosstab — počty

In [5]:
pd.crosstab(
    tips['tip_pct_category'],
    tips['time'],
    margins=True,
    margins_name='Celkem'
)

time,Lunch,Dinner,Celkem
tip_pct_category,,,
skoupý,4,23,27
standard,52,126,178
štědrý,12,27,39
Celkem,68,176,244


### D) Crosstab — průměrné spropitné v $ podle `bill_category` a `day`

In [6]:
pd.crosstab(
    tips['bill_category'],
    tips['day'],
    values=tips['tip'],
    aggfunc='mean'
).round(2)

day,Thur,Fri,Sat,Sun
bill_category,,,,
levný účet,1.90,1.96,1.91,2.48
střední účet,3.12,3.28,3.10,3.33
drahý účet,5.37,4.73,4.81,4.09


### E) Pozorování

**V dolarech (D):** ano, drahé účty mají spropitné výrazně vyšší v $ (~4–5 $) než levné (~2 $). To dává smysl, protože spropitné je obvykle *procento* z účtu.

**V procentech (z C):** rozdělení vypadá přibližně podobně napříč Lunch i Dinner — většina spropitných spadá do kategorie *standard* (10–20 %). V poměru k účtu tedy lidé od velikosti účtu spropitné moc neupravují.

Závěr: čím vyšší účet, tím vyšší **absolutní** spropitné — ale **relativní procento** zůstává podobné.

---
## 2. 🐧 Penguins

In [7]:
penguins = sns.load_dataset('penguins')
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


### A) Binning váhy

In [8]:
bins = [0, 3500, 4500, 5500, float('inf')]
labels = ['velmi malý', 'malý', 'velký', 'velmi velký']
penguins['size_category'] = pd.cut(penguins['body_mass_g'], bins=bins, labels=labels)

penguins['size_category'].value_counts().sort_index()

size_category
velmi malý      78
malý           149
velký           87
velmi velký     28
Name: count, dtype: int64

### B) Crosstab — druhy podle ostrovů

In [9]:
pd.crosstab(
    penguins['species'],
    penguins['island'],
    margins=True,
    margins_name='Celkem'
)

island,Biscoe,Dream,Torgersen,Celkem
species,,,,
Adelie,44,56,52,152
Chinstrap,0,68,0,68
Gentoo,124,0,0,124
Celkem,168,124,52,344


### C) Procentuální struktura ostrovů

In [10]:
pd.crosstab(
    penguins['species'],
    penguins['island'],
    normalize='columns'
).round(3)

island,Biscoe,Dream,Torgersen
species,,,
Adelie,0.262,0.452,1.0
Chinstrap,0.000,0.548,0.0
Gentoo,0.738,0.000,0.0


**Pozorování:** Torgersen má 100 % Adelie tučňáků — jiné druhy tam nežijí. Biscoe je naopak skoro celé Gentoo (~74 %). Dream má smíšenou populaci Adelie a Chinstrap. Některé druhy jsou tedy striktně vázané na konkrétní ostrov.

### D) Průměrná délka ploutví podle druhu a pohlaví

*Pozor:* v datasetu jsou hodnoty `sex` velkými písmeny — `MALE` / `FEMALE`.

In [11]:
pd.crosstab(
    penguins['species'],
    penguins['sex'],
    values=penguins['flipper_length_mm'],
    aggfunc='mean'
).round(1)

sex,Female,Male
species,,
Adelie,187.8,192.4
Chinstrap,191.7,199.9
Gentoo,212.7,221.5


**Pozorování:** samci mají u všech druhů delší ploutve než samice (pohlavní dimorfismus). Gentoo má jasně největší ploutve — víc než 210 mm. Naopak Adelie nejmenší.

### E) Velikostní kategorie podle druhu

In [12]:
pd.crosstab(penguins['size_category'], penguins['species'])

species,Adelie,Chinstrap,Gentoo
size_category,,,
velmi malý,59,19,0
malý,85,47,17
velký,7,2,78
velmi velký,0,0,28


**Pozorování:** Gentoo dominuje *velkým* a *velmi velkým* kategoriím — všech 28 *velmi velkých* tučňáků patří ke Gentoo. Adelie naopak ovládá *velmi malou* a *malou* kategorii. Chinstrap je převážně střední velikost. Velikost tedy spolehlivě poznáš druh.

---
## 3. 🚗 MPG

In [13]:
mpg = sns.load_dataset('mpg')
mpg.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


### A) Binning úspornosti

In [14]:
bins_mpg = [0, 18, 28, float('inf')]
labels_mpg = ['žrout', 'průměr', 'úsporné']
mpg['economy'] = pd.cut(mpg['mpg'], bins=bins_mpg, labels=labels_mpg)

mpg['economy'].value_counts().sort_index()

economy
žrout      124
průměr     166
úsporné    108
Name: count, dtype: int64

### B) Binning éry

*Proč začínáme hranicí 69 a ne 70?* `pd.cut` má intervaly zleva otevřené (`(69, 73]`), takže nejnižší hodnota 70 by jinak vypadla. S `69` zajistíme, že 70 spadne do prvního intervalu.

In [15]:
bins_year = [69, 73, 77, 82]
labels_year = ['začátek 70s', 'polovina 70s', 'konec 70s a 80s']
mpg['era'] = pd.cut(mpg['model_year'], bins=bins_year, labels=labels_year)

mpg['era'].value_counts().sort_index()

era
začátek 70s        125
polovina 70s       119
konec 70s a 80s    154
Name: count, dtype: int64

### C) Úspornost vs. původ

In [16]:
pd.crosstab(
    mpg['economy'],
    mpg['origin'],
    normalize='columns'
).round(3)

origin,europe,japan,usa
economy,,,
žrout,0.057,0.013,0.478
průměr,0.514,0.354,0.410
úsporné,0.429,0.633,0.112


**Pozorování:** USA vyráběly téměř 48 % aut v kategorii *žrout*. Naopak Japonsko mělo *žroutů* jen 1.3 %, zato 63 % úsporných aut. Evropa byla někde uprostřed. Stereotyp o amerických "velkých autech" v datech opravdu sedí.

### D) Průměrný výkon podle éry a původu

In [17]:
pd.crosstab(
    mpg['era'],
    mpg['origin'],
    values=mpg['horsepower'],
    aggfunc='mean'
).round(1)

origin,europe,japan,usa
era,,,
začátek 70s,80.9,90.9,144.3
polovina 70s,83.6,77.4,112.2
konec 70s a 80s,77.1,77.2,98.3


**Pozorování:** v USA výkon klesl dramaticky — ze 144 HP na začátku 70. let na 98 HP ke konci. Naopak v Evropě a Japonsku zůstal výkon prakticky stabilní kolem 77–90 HP. To je pravděpodobně reakce amerických výrobců na ropné krize 70. let a nové emisní standardy.

### E) Bonus — medián váhy podle válců a původu

In [18]:
pd.crosstab(
    mpg['cylinders'],
    mpg['origin'],
    values=mpg['weight'],
    aggfunc='median'
)

origin,europe,japan,usa
cylinders,,,
3,NaN,2375.0,NaN
4,2219.0,2130.0,2408.0
5,2950.0,NaN,NaN
6,3285.0,2905.0,3239.0
8,NaN,NaN,4140.0


**Pozorování:** Některé kombinace neexistují (`NaN`) — třeba **osmiválcová auta vyráběla v tomto období jen USA** a tříválcová jen Japonsko. To je zajímavý poznatek o automobilové diverzitě té doby.

---
## 4. 💎 Diamonds

In [19]:
diamonds = sns.load_dataset('diamonds')
print(diamonds.shape)
diamonds.head()

(53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


### A) Binning ceny

In [20]:
bins_price = [0, 1000, 5000, 15000, float('inf')]
labels_price = ['dostupné', 'střední třída', 'prémium', 'luxus']
diamonds['price_tier'] = pd.cut(diamonds['price'], bins=bins_price, labels=labels_price)

diamonds['price_tier'].value_counts().sort_index()

price_tier
dostupné         14524
střední třída    24702
prémium          13059
luxus             1655
Name: count, dtype: int64

### B) Binning karátů

In [21]:
bins_carat = [0, 0.5, 1.0, 2.0, float('inf')]
labels_carat = ['malý', 'střední', 'velký', 'obří']
diamonds['carat_size'] = pd.cut(diamonds['carat'], bins=bins_carat, labels=labels_carat)

diamonds['carat_size'].value_counts().sort_index()

carat_size
malý       18932
střední    17506
velký      15613
obří        1889
Name: count, dtype: int64

### C) Kvalita brusu vs. cenová třída

In [22]:
pd.crosstab(
    diamonds['cut'],
    diamonds['price_tier'],
    normalize='index'
).round(3)

price_tier,dostupné,střední třída,prémium,luxus
cut,,,,
Ideal,0.318,0.451,0.207,0.025
Premium,0.232,0.426,0.299,0.043
Very Good,0.269,0.455,0.246,0.030
Good,0.227,0.520,0.227,0.026
Fair,0.069,0.665,0.240,0.025


### D) Průměrná cena podle velikosti a brusu

In [23]:
pd.crosstab(
    diamonds['carat_size'],
    diamonds['cut'],
    values=diamonds['price'],
    aggfunc='mean'
).round(0)

cut,Ideal,Premium,Very Good,Good,Fair
carat_size,,,,,
malý,864.0,863.0,766.0,786.0,1028.0
střední,2596.0,2902.0,2959.0,3059.0,2802.0
velký,8044.0,7518.0,7586.0,6935.0,6127.0
obří,15589.0,14992.0,15133.0,14629.0,11972.0


### E) Pozorování — velikost vs. brus

Když porovnáš v tabulce D) *malý Ideal* a *velký Fair*: **velký Fair je řádově dražší** než malý Ideal — i když má horší brus. 

**Závěr:** velikost (karáty) je výrazně silnější faktor ceny než kvalita brusu. Kvalita brusu cenu sice ovlivňuje (uvnitř každé velikostní kategorie je *Ideal* obvykle dražší než *Fair*), ale ten rozdíl je menší než přechod o jednu velikostní kategorii výš.

---
## 5. 🚕 Taxis

In [24]:
taxis = sns.load_dataset('taxis')
taxis.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


### A) Binning vzdálenosti

In [25]:
bins_dist = [0, 1, 3, 10, float('inf')]
labels_dist = ['krátká', 'střední', 'dlouhá', 'velmi dlouhá']
taxis['distance_category'] = pd.cut(taxis['distance'], bins=bins_dist, labels=labels_dist)

taxis['distance_category'].value_counts().sort_index()

distance_category
krátká          1696
střední         2965
dlouhá          1327
velmi dlouhá     394
Name: count, dtype: int64

### B) Binning ceny

In [26]:
bins_total = [0, 10, 25, float('inf')]
labels_total = ['levná jízda', 'střední jízda', 'drahá jízda']
taxis['price_category'] = pd.cut(taxis['total'], bins=bins_total, labels=labels_total)

taxis['price_category'].value_counts().sort_index()

price_category
levná jízda      1252
střední jízda    4114
drahá jízda      1067
Name: count, dtype: int64

### C) Vzdálenost vs. cena

In [27]:
pd.crosstab(taxis['distance_category'], taxis['price_category'])

price_category,levná jízda,střední jízda,drahá jízda
distance_category,,,
krátká,935,757,4
střední,282,2657,26
dlouhá,3,694,630
velmi dlouhá,0,1,393


**Pozorování:** diagonála je opravdu silná — krátké jízdy padají hlavně do *levných*, dlouhé do *drahých*. Tento vztah dává smysl: cena taxi roste s ujetou vzdáleností. Krásný příklad **silné korelace dvou kategorických proměnných**.

### D) Způsob platby podle vzdálenosti

In [28]:
pd.crosstab(
    taxis['distance_category'],
    taxis['payment'],
    normalize='index'
).round(3)

payment,cash,credit card
distance_category,,
krátká,0.334,0.666
střední,0.284,0.716
dlouhá,0.244,0.756
velmi dlouhá,0.170,0.830


**Pozorování:** podíl kartových plateb roste s délkou jízdy (více drahých jízd = lidé radši platí kartou než kupou hotovosti).

### E) Průměrné spropitné podle borough a způsobu platby

In [29]:
pd.crosstab(
    taxis['pickup_borough'],
    taxis['payment'],
    values=taxis['tip'],
    aggfunc='mean'
).round(2)

payment,cash,credit card
pickup_borough,,
Bronx,0.0,0.20
Brooklyn,0.0,1.42
Manhattan,0.0,2.66
Queens,0.0,5.21


**Pozorování — důležitý insight:** sloupec *cash* ukazuje spropitné prakticky **nulové ve všech borough**, zatímco *credit card* má průměrné spropitné kolem 2–3 $. 

**Pozor — to neznamená, že lidé platící hotově nedávají spropitné!** Znamená to, že **hotovostní spropitné se v záznamech taxíku nezachytává** — řidič dostane peníze na ruku a do systému se to nedostane. Karetní spropitné prochází přes platební terminál, takže se nahraje. Datasetu nelze věřit, že odráží realitu — odráží jen, co bylo *zaznamenáno*. 

Tento jev je klasická ukázka **selection bias v datech** — perfektní lekce o tom, že před interpretací výsledků musíš rozumět, jak vznikly.